# Visualisation du Nettoyage des Données (Avant / Après)

Ce notebook utilise `pandas` pour illustrer concrètement l'impact de l'étape de nettoyage automatique sur nos événements OpenAgenda.
Nous allons comparer les données brutes (`events_raw.json`) avec les données propres prêtes pour le RAG (`events_clean.json`).

In [1]:
import pandas as pd
import json

# 1. Chargement des données brutes (Avant nettoyage)
with open('../data/events_raw.json', 'r', encoding='utf-8') as f:
    df_raw = pd.DataFrame(json.load(f))

# 2. Chargement des données propres (Après nettoyage)
with open('../data/events_clean.json', 'r', encoding='utf-8') as f:
    df_clean = pd.DataFrame(json.load(f))

pd.set_option('display.max_rows', None)

print("Données chargées avec succès pour la comparaison.")

Données chargées avec succès pour la comparaison.


In [2]:
df_raw.isnull().sum()

image                                                                                                                                             182
featured                                                                                                                                            0
attendanceMode                                                                                                                                      0
keywords                                                                                                                                            0
dateRange                                                                                                                                           0
timezone                                                                                                                                            0
imageCredits                                                                                        

In [3]:
seuil_tolerance = 0.50 # 50%
limite_valeurs_non_nulles = int((1 - seuil_tolerance) * len(df_raw))

# thresh = "Nombre minimum de valeurs valides requises pour garder la colonne"
df_raw = df_raw.dropna(axis=1, thresh=limite_valeurs_non_nulles)
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 6148 entries, 0 to 6147
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   image              5966 non-null   object
 1   featured           6148 non-null   bool  
 2   attendanceMode     6148 non-null   int64 
 3   keywords           6148 non-null   object
 4   dateRange          6148 non-null   object
 5   timezone           6148 non-null   str   
 6   imageCredits       3462 non-null   str   
 7   originAgenda       6148 non-null   object
 8   description        6148 non-null   object
 9   title              6148 non-null   object
 10  uid                6148 non-null   int64 
 11  lastTiming         6148 non-null   object
 12  firstTiming        6148 non-null   object
 13  location           6119 non-null   object
 14  slug               6148 non-null   str   
 15  status             6148 non-null   int64 
 16  parent_agenda_uid  6148 non-null   int64 
dtypes: boo

In [4]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 5804 entries, 0 to 5803
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   uid             5804 non-null   int64 
 1   slug            5804 non-null   str   
 2   title_fr        5804 non-null   str   
 3   description_fr  5804 non-null   str   
 4   keywords_fr     5804 non-null   object
 5   location        5804 non-null   object
 6   firstTiming     5804 non-null   object
 7   lastTiming      5804 non-null   object
 8   attendanceMode  5804 non-null   int64 
 9   image           5632 non-null   object
dtypes: int64(2), object(5), str(3)
memory usage: 453.6+ KB


## 1. Bilan Général : Doublons et Valeurs Manquantes
Le tableau suivant compare le volume de données, la présence de doublons (basé sur l'identifiant `uid`) et le nombre d'événements sans adresse.

In [5]:
# Calcul des statistiques comparatives
stats_data = {
    "Étape": ["Avant Nettoyage (Brut)", "Après Nettoyage (Propre)"],
    "Nombre total d'événements": [len(df_raw), len(df_clean)],
    "Doublons (UID)": [df_raw.duplicated(subset=['uid']).sum(), df_clean.duplicated(subset=['uid']).sum()],
    "Sans Localisation (Valeurs manquantes)": [df_raw['location'].isnull().sum(), df_clean['location'].isnull().sum()]
}

df_stats = pd.DataFrame(stats_data).set_index("Étape")

# Affichage visuel sous forme de tableau Pandas
df_stats

,Nombre total d'événements,Doublons (UID),Sans Localisation (Valeurs manquantes)
Étape,,,
Avant Nettoyage (Brut),6148,321,29
Après Nettoyage (Propre),5804,0,0


## 2. Valeurs incorrectes ou bruitées : Le problème du multilingue
Dans les données brutes, les champs textuels (titre, description) sont des dictionnaires contenant plusieurs langues (français, anglais, espagnol...). 
Cela constitue des **données incorrectes** ou bruitées pour notre RAG qui ne fonctionne qu'en français. 

Voici la comparaison visuelle d'un champ avant et après extraction de la version française :

In [6]:
print("\n❌ AVANT NETTOYAGE (Champ 'title' complet avec toutes les langues) :")
print(json.dumps(df_raw['title'].iloc[0], indent=2, ensure_ascii=False))

print("\n" + "-"*50)

print("\n✅ APRÈS NETTOYAGE (Champ 'title_fr' ciblé) :")
print(df_clean['title_fr'].iloc[0])


❌ AVANT NETTOYAGE (Champ 'title' complet avec toutes les langues) :
{
  "fr": "Webinaire – La réforme de la facturation électronique : quels changements pour mon exploitation ?"
}

--------------------------------------------------

✅ APRÈS NETTOYAGE (Champ 'title_fr' ciblé) :
Webinaire – La réforme de la facturation électronique : quels changements pour mon exploitation ?


## 3. Séparation du "Signal" sémantique et du "Bruit" technique
Même après avoir retiré les colonnes très vides, il nous restait des colonnes bien remplies mais inutiles pour le RAG (ex: `slug`, `timezone`, `imageCredits`).

**Pourquoi les supprimer ?** Pour optimiser les vecteurs mathématiques (Embeddings) de FAISS. On veut concentrer le modèle uniquement sur le sens du texte, sans polluer le calcul de similarité avec des mots-clés administratifs.

In [7]:
# Comparaison des colonnes (Signal vs Bruit)
colonnes_techniques_supprimees = ['slug', 'timezone', 'updatedAt', 'originAgenda', 'imageCredits']
colonnes_semantiques_conservees = ['title_fr', 'description_fr', 'location', 'keywords_fr', 'image', 'firstTiming', 'lastTiming']

df_explication = pd.DataFrame({
    "Type de Donnée": ["Bruit Technique (Supprimé)", "Signal Sémantique (Conservé)"],
    "Exemples de colonnes": [
        ", ".join(colonnes_techniques_supprimees),
        ", ".join(colonnes_semantiques_conservees)
    ],
    "Impact sur le RAG": [
        "Dilue la similarité vectorielle (FAISS) et consomme des tokens LLM inutilement.",
        "Concentre la recherche sur le vrai sens de l'événement."
    ]
})

df_explication.set_index("Type de Donnée")

,Exemples de colonnes,Impact sur le RAG
Type de Donnée,,
Bruit Technique (Supprimé),"slug, timezone, updatedAt, originAgenda, image...",Dilue la similarité vectorielle (FAISS) et con...
Signal Sémantique (Conservé),"title_fr, description_fr, location, keywords_f...",Concentre la recherche sur le vrai sens de l'é...


## 4. Aperçu visuel des données prêtes pour l'indexation FAISS
Voici à quoi ressemblent les premières lignes de notre DataFrame final, propre, unifié et prêt à être injecté dans notre base vectorielle.

In [8]:
cols_to_display = ['uid', 'title_fr', 'description_fr', 'location', 'firstTiming', 'lastTiming', 'image']
df_clean[cols_to_display].head()

,uid,title_fr,description_fr,location,firstTiming,lastTiming,image
0,89578114,Webinaire – La réforme de la facturation élect...,La CA07 organise un webinaire pour comprendre ...,"{'address': '07000', 'city': 'Veyras', 'latitu...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...",{'filename': 'e08c6495a6b648b5a93fe5373cc94c5e...
1,65182490,Webinaire Castanea Sativa,La Chambre d'agriculture de l'Ardèch propose u...,"{'address': '07000', 'city': 'Veyras', 'latitu...","{'end': '2026-07-08T05:00:00.000+02:00', 'begi...","{'end': '2026-07-08T05:00:00.000+02:00', 'begi...",{'filename': '420f5057b6ee465cbd93bc8a5865bed6...
2,36521668,Estive en fête !,Randonnée pastorale festive sur les hauteurs d...,"{'address': 'la croix de Bauzon, Ardèche', 'ci...","{'end': '2026-07-11T16:00:00.000+02:00', 'begi...","{'end': '2026-07-11T16:00:00.000+02:00', 'begi...",{'filename': 'cfbec81589874586b7405bf429f19d35...
3,95768697,Journée Technique Hydrologie regénérative en f...,demi-journée technique gratuite et ouverte à t...,"{'address': '2, place de la salle des fêtes 07...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...",{'filename': '05ec99b07dcc48eca79924b4d824db9d...
4,36284787,Facturation électronique,Réunion d’information interconsulaire sur la m...,"{'address': 'coucouron', 'city': 'Coucouron', ...","{'end': '2026-06-01T17:00:00.000+02:00', 'begi...","{'end': '2026-06-01T17:00:00.000+02:00', 'begi...",{'filename': 'ea0f08b6147a482ba09bcd85bc93e4e1...
